# IBM AML Demo Queries

This notebook runs Gremlin queries and shows query results.
Start backend first in another terminal:
`mvn exec:java`


In [1]:
import requests
import pandas as pd
import subprocess
import json as _json
from typing import Dict, Any
from IPython.display import display, Markdown
from pathlib import Path

BASE_URL = "http://localhost:7000"
MAX_ROWS = 100000
REPO_ROOT = next((p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / "pom.xml").exists()), Path.cwd())
SHOW_PLAN = False


def run_aml_data_download(variant: str = "HI-Small", rows: int = 100000) -> bool:
    """
    Download the AML dataset from Kaggle, normalise to engine format, and seed H2.
    All three stages use *rows* as the row limit — MAX_ROWS controls everything.

    Uses prepare_aml_data.py which:
      1. Downloads raw CSV from Kaggle (skips if already present locally)
      2. Normalises to engine format writing exactly *rows* rows
      3. Bulk-loads into H2 via CSVREAD() — one INSERT per table
    """
    prepare_script = REPO_ROOT / "demo" / "aml" / "scripts" / "prepare_aml_data.py"
    if not prepare_script.exists():
        display(Markdown(f"**prepare_aml_data.py not found:** `{prepare_script}`"))
        return False

    cmd = [
        "python3", str(prepare_script),
        "--variant", variant,
        "--rows",    str(rows),
        "--backend", "h2",
    ]
    display(Markdown(
        f"Preparing AML data (Kaggle download → normalise → H2 seed, MAX_ROWS={rows:,})\n\n"
        f"`{' '.join(str(c) for c in cmd)}`"
    ))
    # Use inherited stdout/stderr so progress streams live to the notebook.
    proc = subprocess.run(cmd, cwd=str(REPO_ROOT))
    if proc.returncode != 0:
        display(Markdown(
            f"**prepare failed** (exit {proc.returncode}) — see output above."
        ))
        return False
    return True


def resolve_raw_source_csv() -> str | None:
    candidates = [
        Path.cwd() / "demo/aml/data/HI-Small_Trans.csv",
        REPO_ROOT / "demo/aml/data/HI-Small_Trans.csv",
    ]
    for p in candidates:
        if p.exists():
            return str(p)

    demo_data = REPO_ROOT / "demo/aml/data"
    if demo_data.exists():
        matches = sorted(demo_data.glob("HI-*_Trans.csv"))
        if matches:
            return str(matches[0])
    return None


def normalized_csv_path() -> str:
    return str(REPO_ROOT / "demo/aml/data/aml-demo.csv")


def _count_rows_upto(csv_path: str, max_rows: int) -> int:
    import csv
    rows = 0
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)
        for _ in reader:
            rows += 1
            if rows >= max_rows:
                break
    return rows


def normalize_for_max_rows(src_csv: str, max_rows: int) -> str | None:
    out_csv = normalized_csv_path()
    normalize_py = REPO_ROOT / "scripts/normalize_aml.py"
    if not normalize_py.exists():
        display(Markdown(f"**normalize_aml.py missing:** `{normalize_py}`"))
        return None

    cmd = [
        "python3",
        str(normalize_py),
        "--src",
        str(src_csv),
        "--dst",
        str(out_csv),
        "--rows",
        str(max_rows),
    ]
    display(Markdown(f"Preparing normalized CSV for MAX_ROWS via: `{ ' '.join(cmd) }`"))
    proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        return None
    return out_csv if Path(out_csv).exists() else None


def resolve_mapping_path() -> str | None:
    candidates = [
        Path.cwd() / "demo/aml/mappings/aml-mapping.json",
        REPO_ROOT / "demo/aml/mappings/aml-mapping.json",
    ]
    for p in candidates:
        if p.exists():
            return str(p)
    return None


CSV_PATH = normalized_csv_path() if Path(normalized_csv_path()).exists() else None
RAW_SOURCE_CSV = resolve_raw_source_csv()
MAPPING_PATH = resolve_mapping_path()
print("BASE_URL:", BASE_URL)
print("CSV_PATH:", CSV_PATH)
print("RAW_SOURCE_CSV:", RAW_SOURCE_CSV)
print("MAPPING_PATH:", MAPPING_PATH)
print("MAX_ROWS:", MAX_ROWS)


def try_upload_mapping() -> bool:
    if not MAPPING_PATH:
        display(Markdown("**AML mapping not found.** Expected `mappings/aml-mapping.json` or `demo/aml/mappings/aml-mapping.json`."))
        return False

    try:
        with open(MAPPING_PATH, "rb") as mapping_file:
            mapping_resp = requests.post(
                f"{BASE_URL}/mapping/upload",
                files={"file": mapping_file},
                timeout=QUERY_TIMEOUT,
            )
    except Exception as e:
        display(Markdown(f"**Mapping upload failed:** {e}"))
        return False

    if mapping_resp.status_code not in (200, 201):
        display(Markdown(f"**Mapping upload failed** (`HTTP {mapping_resp.status_code}`):"))
        print(mapping_resp.text)
        return False

    print(mapping_resp.json())
    return True


def get_account_count() -> int | None:
    try:
        count_resp = requests.post(
            f"{BASE_URL}/gremlin/query",
            json={"gremlin": "g.V().hasLabel('Account').count()"},
            timeout=QUERY_TIMEOUT,
        )
        if count_resp.status_code != 200:
            display(Markdown(f"**Data verification failed** (`HTTP {count_resp.status_code}`):"))
            print(count_resp.text)
            return None

        payload = count_resp.json()
        values = payload.get("results") or []
        if not values:
            return 0
        return int(values[0])
    except Exception as e:
        display(Markdown(f"**Data verification error:** {e}"))
        return None


def get_transfer_count() -> int | None:
    try:
        count_resp = requests.post(
            f"{BASE_URL}/gremlin/query",
            json={"gremlin": "g.E().hasLabel('TRANSFER').count()"},
            timeout=QUERY_TIMEOUT,
        )
        if count_resp.status_code != 200:
            display(Markdown(f"**Transfer count check failed** (`HTTP {count_resp.status_code}`):"))
            print(count_resp.text)
            return None

        payload = count_resp.json()
        values = payload.get("results") or []
        if not values:
            return 0
        return int(values[0])
    except Exception as e:
        display(Markdown(f"**Transfer count check error:** {e}"))
        return None


def count_csv_rows(csv_path: str, max_rows: int) -> int:
    return _count_rows_upto(csv_path, max_rows)


def run_aml_loader_script(csv_path: str, max_rows: int) -> bool:
    if not MAPPING_PATH:
        display(Markdown("**AML mapping not found.** Expected `mappings/aml-mapping.json` or `demo/aml/mappings/aml-mapping.json`."))
        return False

    if not REPO_ROOT.exists():
        display(Markdown(f"**Error:** repo path not found: `{REPO_ROOT}`"))
        return False

    cmd = [
        "python3",
        "demo/aml/aml_csv_loader.py",
        "--path",
        str(csv_path),
        "--max-rows",
        str(max_rows),
        "--url",
        BASE_URL,
        "--mapping",
        str(MAPPING_PATH),
    ]
    display(Markdown(f"Running loader script: `{ ' '.join(cmd) }`"))
    proc = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    return proc.returncode == 0


def ensure_csv_present(auto_download: bool = True, max_rows: int = 100_000) -> str | None:
    global CSV_PATH, RAW_SOURCE_CSV

    if CSV_PATH and Path(CSV_PATH).exists():
        existing = _count_rows_upto(CSV_PATH, max_rows)
        if existing >= max_rows:
            return CSV_PATH

    if not RAW_SOURCE_CSV:
        RAW_SOURCE_CSV = resolve_raw_source_csv()

    if not RAW_SOURCE_CSV and auto_download:
        ok = run_aml_data_download(variant="HI-Small", rows=max_rows)
        if ok:
            RAW_SOURCE_CSV = resolve_raw_source_csv()

    if not RAW_SOURCE_CSV:
        return None

    prepared = normalize_for_max_rows(RAW_SOURCE_CSV, max_rows)
    if prepared:
        CSV_PATH = prepared
    return CSV_PATH


def ensure_aml_ready(auto_download: bool = True, max_rows: int = 100_000) -> None:
    csv_path = ensure_csv_present(auto_download=auto_download, max_rows=max_rows)
    if not csv_path:
        display(Markdown("**CSV not found after download attempt.**"))
        return

    display(Markdown(f"CSV detected: `{csv_path}`"))

    account_count = get_account_count()
    if account_count is None:
        return

    transfer_count = get_transfer_count()
    if transfer_count is None:
        return

    expected_rows = count_csv_rows(csv_path, max_rows)
    display(Markdown(f"CSV rows considered (up to MAX_ROWS): **{expected_rows}**"))
    display(Markdown(f"Current graph TRANSFER edges: **{transfer_count}**"))

    if transfer_count == expected_rows and expected_rows > 0:
        # Keep mapping in sync for query/explain even when data already exists.
        try_upload_mapping()
        display(Markdown(f"AML data already loaded and up-to-date (`TRANSFER={transfer_count}`)."))
        return

    display(Markdown("Dataset not loaded (or row count mismatch); running AML loader script..."))
    loaded = run_aml_loader_script(csv_path, max_rows)
    if not loaded:
        display(Markdown("**Loader script failed.** Check output above."))
        return

    refreshed_accounts = get_account_count()
    refreshed_transfers = get_transfer_count()
    if refreshed_accounts is not None and refreshed_transfers is not None:
        display(Markdown(
            f"AML data load complete: **{refreshed_accounts}** Account vertices, "
            f"**{refreshed_transfers}** TRANSFER edges."
        ))
QUERY_TIMEOUT = 120  # seconds — increase for slow/complex queries

# ── Execution backend ─────────────────────────────────────────────────────────
# The engine supports three GRAPH_PROVIDER modes (set when starting the server):
#
#   GRAPH_PROVIDER=sql    (default) Gremlin → SQL → JDBC
#   GRAPH_PROVIDER=wcoj   Worst-Case Optimal Join engine — uses in-memory
#                         adjacency index + Leapfrog Trie Join for multi-hop
#                         path queries; auto-falls-back to SQL for aggregations
#                         Start: GRAPH_PROVIDER=wcoj mvn exec:java
#   GRAPH_PROVIDER=tinkergraph  Native TinkerPop (dev/testing only)
#
# The notebook always calls /gremlin/query — the provider is transparent.
# Switch provider by restarting the server with a different GRAPH_PROVIDER.
# ──────────────────────────────────────────────────────────────────────────────


BASE_URL: http://localhost:7000
CSV_PATH: None
RAW_SOURCE_CSV: None
MAPPING_PATH: None
MAX_ROWS: 100000


## Step 1: Prepare CSV (HI-Small)

Step 2 reuses existing `HI-Small_Trans.csv` and normalizes `aml-demo.csv` for the current `MAX_ROWS`.
If raw HI-Small files are missing, run this in terminal:

```zsh
cd ~/SourceCode/graph-query-engine
bash ./scripts/download_aml_data.sh --variant HI-Small --rows 100000
```

This downloads `HI-Small_Trans.csv` and prepares `demo/data/aml-demo.csv`.

You can also run the helper below from notebook.


In [2]:
AUTO_RUN_DOWNLOAD = False
DOWNLOAD_VARIANT = "HI-Small"
DOWNLOAD_ROWS = 100000

if AUTO_RUN_DOWNLOAD:
    ok = run_aml_data_download(variant=DOWNLOAD_VARIANT, rows=DOWNLOAD_ROWS)
    if ok:
        RAW_SOURCE_CSV = resolve_raw_source_csv()
        CSV_PATH = normalize_for_max_rows(RAW_SOURCE_CSV, DOWNLOAD_ROWS) if RAW_SOURCE_CSV else None
        print("CSV_PATH refreshed:", CSV_PATH)
else:
    print("Set AUTO_RUN_DOWNLOAD=True to run download + normalize from notebook.")


Set AUTO_RUN_DOWNLOAD=True to run download + normalize from notebook.


In [3]:
def get_sql_explain(gremlin_query: str, include_plan: bool = False) -> Dict[str, Any]:
    try:
        response = requests.post(
            f"{BASE_URL}/query/explain",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json"},
            params={"plan": "true"} if include_plan else {},
            timeout=10,
        )
        if response.ok:
            return response.json()
        return {"error": f"HTTP {response.status_code}: {response.text}"}
    except Exception as e:
        return {"error": str(e)}


def run_gremlin_query(gremlin_query: str, tx_mode: bool = False) -> Dict[str, Any]:
    endpoint = "/gremlin/query/tx" if tx_mode else "/gremlin/query"
    result: Dict[str, Any] = {}
    try:
        response = requests.post(
            f"{BASE_URL}{endpoint}",
            json={"gremlin": gremlin_query},
            headers={"Content-Type": "application/json"},
            timeout=60,
        )
        result = response.json()
    except Exception as e:
        result = {"error": str(e)}
    result["_sql_explain"] = get_sql_explain(gremlin_query, include_plan=SHOW_PLAN)
    return result


def display_query_result(gremlin: str, result: Dict[str, Any], title: str = "", limit: int = 10, tx_mode: bool = False):
    if title:
        display(Markdown(f"### {title}"))
    display(Markdown("**Gremlin:**"))
    display(Markdown(f"```groovy\n{gremlin}\n```"))

    explain = result.get("_sql_explain", {})
    if "error" in explain:
        display(Markdown(f"**SQL Translation:** *not available ({explain['error']})*"))
    else:
        sql = explain.get("translatedSql", "")
        params = explain.get("parameters", [])
        plan = explain.get("plan")
        if sql:
            display(Markdown("**SQL Translation:**"))
            display(Markdown(f"```sql\n{sql}\n```"))
            if params:
                display(Markdown(f"**Parameters:** `{params}`"))
        if plan:
            display(Markdown("**Query Plan:**"))
            display(Markdown(f"```json\n{_json.dumps(plan, indent=2)}\n```"))

    if "error" in result:
        display(Markdown(f"**Error:** {result['error']}"))
        return

    rows = result.get("results", [])
    display(Markdown(f"**Result Count:** {result.get('resultCount', 0)}"))
    if not rows:
        return
    if isinstance(rows[0], dict):
        display(pd.DataFrame(rows[:limit]))
    else:
        for i, row in enumerate(rows[:limit], 1):
            print(f"{i}. {row}")


## Step 0: Health check


In [4]:
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5).text
    provider = requests.get(f"{BASE_URL}/gremlin/provider", timeout=5).json().get("provider", "unknown")
    display(Markdown(f"Status: `{health}`"))
    display(Markdown(f"Provider: `{provider}`"))
except Exception as e:
    display(Markdown(f"Health check failed: {e}"))


Status: `{"status":"ok","service":"graph-query-engine","dbUrl":"jdbc:h2:file:./data/graph;AUTO_SERVER=TRUE","provider":"sql-multi","backends":[{"url":"jdbc:h2:file:./data/graph;AUTO_SERVER=TRUE","id":"default"},{"url":"jdbc:trino://localhost:8080/iceberg","id":"iceberg"}],"defaultBackend":"default"}`

Provider: `sql-multi`

## Step 2: Validate CSV, Upload Mapping, Verify Data


In [5]:
AUTO_DOWNLOAD_IF_MISSING = True
ensure_aml_ready(auto_download=AUTO_DOWNLOAD_IF_MISSING, max_rows=MAX_ROWS)


Running: `bash ./scripts/download_aml_data.sh --variant HI-Small --rows 100000` in `/Users/vjoshi/SourceCode/graph-query-engine`

bash: ./scripts/download_aml_data.sh: No such file or directory



**CSV not found after download attempt.**

## Query Sections


## Simple Queries


### S1 Count accounts

Count unique Account vertices.


In [6]:
gremlin = "g.V().hasLabel('Account').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S1 Count accounts', limit=10, tx_mode=False)


### S1 Count accounts

**Gremlin:**

```groovy
g.V().hasLabel('Account').count()
```

**SQL Translation:** *not available (HTTP 400: {"error":"No vertex mapping found for label: Account"})*

**Error:** SQL translation failed: No vertex mapping found for label: Account

### S2 Count banks

Count Bank vertices - one per distinct bank ID in the data.


In [7]:
gremlin = "g.V().hasLabel('Bank').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S2 Count banks', limit=10, tx_mode=False)


### S2 Count banks

**Gremlin:**

```groovy
g.V().hasLabel('Bank').count()
```

**SQL Translation:** *not available (HTTP 400: {"error":"No vertex mapping found for label: Bank"})*

**Error:** SQL translation failed: No vertex mapping found for label: Bank

### S3 Count countries

Count Country vertices (10 pre-seeded jurisdictions).


In [8]:
gremlin = "g.V().hasLabel('Country').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S3 Count countries', limit=10, tx_mode=False)


### S3 Count countries

**Gremlin:**

```groovy
g.V().hasLabel('Country').count()
```

**SQL Translation:** *not available (HTTP 400: {"error":"No vertex mapping found for label: Country"})*

**Error:** SQL translation failed: No vertex mapping found for label: Country

### S4 Count alerts

Count Alert vertices - one raised per suspicious transfer.


In [9]:
gremlin = "g.V().hasLabel('Alert').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S4 Count alerts', limit=10, tx_mode=False)


### S4 Count alerts

**Gremlin:**

```groovy
g.V().hasLabel('Alert').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_alerts
```

**Result Count:** 1

1. 1


### S5 Count transfers

Count all TRANSFER edges.


In [10]:
gremlin = "g.E().hasLabel('TRANSFER').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S5 Count transfers', limit=10, tx_mode=False)


### S5 Count transfers

**Gremlin:**

```groovy
g.E().hasLabel('TRANSFER').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_transfers
```

**Result Count:** 1

1. 100000


### S6 Suspicious transfer count

Count confirmed suspicious TRANSFER edges (isLaundering=1).


In [11]:
gremlin = "g.E().has('isLaundering','1').count()"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S6 Suspicious transfer count', limit=10, tx_mode=False)


### S6 Suspicious transfer count

**Gremlin:**

```groovy
g.E().has('isLaundering','1').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_transfers WHERE is_laundering = ?
```

**Parameters:** `['1']`

**Result Count:** 1

1. 5


### S7 High-risk countries

List Country vertices with riskLevel=HIGH.


In [12]:
gremlin = "g.V().hasLabel('Country').has('riskLevel','HIGH').project('countryCode','countryName','region','fatfBlacklist').by('countryCode').by('countryName').by('region').by('fatfBlacklist')"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S7 High-risk countries', limit=10, tx_mode=False)


### S7 High-risk countries

**Gremlin:**

```groovy
g.V().hasLabel('Country').has('riskLevel','HIGH').project('countryCode','countryName','region','fatfBlacklist').by('countryCode').by('countryName').by('region').by('fatfBlacklist')
```

**SQL Translation:**

```sql
SELECT v.country_code AS "countryCode", v.country_name AS "countryName", v.region AS "region", v.fatf_blacklist AS "fatfBlacklist" FROM aml_countries v WHERE v.risk_level = ?
```

**Parameters:** `['HIGH']`

**Result Count:** 3

,countryCode,countryName,region,fatfBlacklist
0,NG,Nigeria,Africa,false
1,KY,Cayman Islands,Americas,true
2,PA,Panama,Americas,true


### S8 High-severity alerts

Show HIGH-severity open alerts.


In [13]:
gremlin = "g.V().hasLabel('Alert').has('severity','HIGH').project('alertId','alertType','status','raisedAt').by('alertId').by('alertType').by('status').by('raisedAt').limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='S8 High-severity alerts', limit=15, tx_mode=False)


### S8 High-severity alerts

**Gremlin:**

```groovy
g.V().hasLabel('Alert').has('severity','HIGH').project('alertId','alertType','status','raisedAt').by('alertId').by('alertType').by('status').by('raisedAt').limit(15)
```

**SQL Translation:**

```sql
SELECT v.alert_id AS "alertId", v.alert_type AS "alertType", v.status AS "status", v.raised_at AS "raisedAt" FROM aml_alerts v WHERE v.severity = ? LIMIT 15
```

**Parameters:** `['HIGH']`

**Result Count:** 0

## Complex Queries


### C1 Top sender accounts

Accounts ranked by outgoing transfer count - find the biggest hubs.


In [14]:
gremlin = "g.V().hasLabel('Account').project('accountId','bankId','outDegree').by('accountId').by('bankId').by(outE('TRANSFER').count()).order().by(select('outDegree'),Order.desc).limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C1 Top sender accounts', limit=15, tx_mode=False)


### C1 Top sender accounts

**Gremlin:**

```groovy
g.V().hasLabel('Account').project('accountId','bankId','outDegree').by('accountId').by('bankId').by(outE('TRANSFER').count()).order().by(select('outDegree'),Order.desc).limit(15)
```

**SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id) AS "outDegree" FROM aml_accounts v ORDER BY outDegree DESC LIMIT 15
```

**Result Count:** 15

,accountId,bankId,outDegree
0,100428660,070,1924
1,8001409E0,001,13
2,80026D340,010,13
3,800631890,01601,13
4,8001C3570,010,12
5,8008F0D20,021174,12
6,8001523C0,010,11
7,8001A4930,012,11
8,8001F3760,010,11
9,800537DE0,001,11


### C2 Suspicious hubs

Accounts with the most suspicious outgoing transfers.


In [15]:
gremlin = "g.V().hasLabel('Account').project('accountId','bankId','suspiciousOut','totalOut').by('accountId').by('bankId').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).where(select('suspiciousOut').is(gt(0))).order().by(select('suspiciousOut'),Order.desc).limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C2 Suspicious hubs', limit=15, tx_mode=False)


### C2 Suspicious hubs

**Gremlin:**

```groovy
g.V().hasLabel('Account').project('accountId','bankId','suspiciousOut','totalOut').by('accountId').by('bankId').by(outE('TRANSFER').has('isLaundering','1').count()).by(outE('TRANSFER').count()).where(select('suspiciousOut').is(gt(0))).order().by(select('suspiciousOut'),Order.desc).limit(15)
```

**SQL Translation:**

```sql
SELECT * FROM (SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id AND is_laundering = '1') AS "suspiciousOut", (SELECT COUNT(*) FROM aml_transfers WHERE from_account_id = v.id) AS "totalOut" FROM aml_accounts v) p WHERE p."suspiciousOut" > 0 ORDER BY suspiciousOut DESC LIMIT 15
```

**Result Count:** 1

,accountId,bankId,suspiciousOut,totalOut
0,100428660,070,5,1924


### C3 Account -> Bank (BELONGS_TO)

Show which bank each account belongs to via BELONGS_TO.


In [16]:
gremlin = "g.V().hasLabel('Account').limit(15).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C3 Account -> Bank (BELONGS_TO)', limit=15, tx_mode=False)


### C3 Account -> Bank (BELONGS_TO)

**Gremlin:**

```groovy
g.V().hasLabel('Account').limit(15).project('accountId','bankId','bankName').by('accountId').by('bankId').by(out('BELONGS_TO').values('bankName').fold())
```

**SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT STRING_AGG(_njv0.bank_name, ',') FROM aml_account_bank _nje0 JOIN aml_banks _njv0 ON _njv0.id = _nje0.bank_id WHERE _nje0.account_id = v.id) AS "bankName" FROM aml_accounts v LIMIT 15
```

**Result Count:** 15

,accountId,bankId,bankName
0,806758E70,0115700,[Bank-0115700]
1,80C15ED80,028255,[Bank-028255]
2,8019773B0,032390,[Bank-032390]
3,8067598C0,009129,[Bank-009129]
4,80C163090,0115249,[Bank-0115249]
5,80E3BCFC0,016031,[Bank-016031]
6,801977590,021174,[Bank-021174]
7,806759B30,0017246,[Bank-0017246]
8,803BF4980,007478,[Bank-007478]
9,80E3BE180,0038486,[Bank-0038486]


### C4 Bank -> Country (LOCATED_IN)

Show which country each bank is headquartered in.


In [17]:
gremlin = "g.V().hasLabel('Bank').limit(15).project('bankId','bankName','countryCode','countryName').by('bankId').by('bankName').by('countryCode').by(out('LOCATED_IN').values('countryName').fold())"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C4 Bank -> Country (LOCATED_IN)', limit=15, tx_mode=False)


### C4 Bank -> Country (LOCATED_IN)

**Gremlin:**

```groovy
g.V().hasLabel('Bank').limit(15).project('bankId','bankName','countryCode','countryName').by('bankId').by('bankName').by('countryCode').by(out('LOCATED_IN').values('countryName').fold())
```

**SQL Translation:**

```sql
SELECT v.bank_id AS "bankId", v.bank_name AS "bankName", v.country_code AS "countryCode", (SELECT STRING_AGG(_njv0.country_name, ',') FROM aml_bank_country _nje0 JOIN aml_countries _njv0 ON _njv0.id = _nje0.country_id WHERE _nje0.bank_id = v.id) AS "countryName" FROM aml_banks v LIMIT 15
```

**Result Count:** 15

,bankId,bankName,countryCode,countryName
0,010,Bank-010,SG,[Singapore]
1,0236746,Bank-0236746,HK,[Hong Kong]
2,03208,Bank-03208,CH,[Switzerland]
3,001,Bank-001,SG,[Singapore]
4,0338838,Bank-0338838,PA,[Panama]
5,03209,Bank-03209,HK,[Hong Kong]
6,012,Bank-012,NG,[Nigeria]
7,002439,Bank-002439,AE,[UAE]
8,0336009,Bank-0336009,NG,[Nigeria]
9,0211050,Bank-0211050,SG,[Singapore]


### C5 Two-hop: Account -> Bank -> Country

Traverse Account->Bank->Country in two hops.


In [18]:
gremlin = "g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C5 Two-hop: Account -> Bank -> Country', limit=10, tx_mode=False)


### C5 Two-hop: Account -> Bank -> Country

**Gremlin:**

```groovy
g.V().hasLabel('Account').limit(1).repeat(out('BELONGS_TO','LOCATED_IN').simplePath()).times(2).path().by('accountId').by('bankName').by('countryName').limit(10)
```

**SQL Translation:**

```sql
SELECT v0.account_id AS accountId0, v1.bank_name AS bankName1, v2.country_name AS countryName2 FROM (SELECT * FROM aml_accounts LIMIT 1) v0 JOIN aml_account_bank e1 ON e1.account_id = v0.id JOIN aml_banks v1 ON v1.id = e1.bank_id JOIN aml_bank_country e2 ON e2.bank_id = v1.id JOIN aml_countries v2 ON v2.id = e2.country_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) LIMIT 10
```

**Result Count:** 1

1. ['806758E70', 'Bank-0115700', 'United States']


### C6 Accounts sending to high-risk countries (SENT_VIA)

Find accounts routing money via FATF-blacklisted countries.


In [19]:
gremlin = "g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(20)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C6 Accounts sending to high-risk countries (SENT_VIA)', limit=20, tx_mode=False)


### C6 Accounts sending to high-risk countries (SENT_VIA)

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(out('SENT_VIA').has('fatfBlacklist','true')).project('accountId','bankId','riskScore').by('accountId').by('bankId').by('riskScore').limit(20)
```

**SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", v.risk_score AS "riskScore" FROM aml_accounts v WHERE EXISTS (SELECT 1 FROM aml_transfer_channel _we JOIN aml_countries _wv ON _wv.id = _we.country_id WHERE _we.transfer_id = v.id AND _wv.fatf_blacklist = ?) LIMIT 20
```

**Parameters:** `['true']`

**Result Count:** 20

,accountId,bankId,riskScore
0,8067598C0,009129,22
1,80C163090,0115249,82
2,80E3BCFC0,016031,46
3,80925E3C0,0221891,42
4,801977D30,01665,45
5,803BF9460,001686,154
6,80196EC80,011405,98
7,803C029A0,027444,126
8,80675C660,01467,17
9,80196F310,011405,70


### C7 Flagged accounts with alert detail (FLAGGED_BY)

Show investigated accounts with linked Alert severity.


In [20]:
gremlin = "g.V().hasLabel('Account').where(outE('FLAGGED_BY')).project('accountId','bankId','alertCount','highSeverity').by('accountId').by('bankId').by(outE('FLAGGED_BY').count()).by(out('FLAGGED_BY').has('severity','HIGH').count()).limit(20)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C7 Flagged accounts with alert detail (FLAGGED_BY)', limit=20, tx_mode=False)


### C7 Flagged accounts with alert detail (FLAGGED_BY)

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(outE('FLAGGED_BY')).project('accountId','bankId','alertCount','highSeverity').by('accountId').by('bankId').by(outE('FLAGGED_BY').count()).by(out('FLAGGED_BY').has('severity','HIGH').count()).limit(20)
```

**SQL Translation:**

```sql
SELECT v.account_id AS "accountId", v.bank_id AS "bankId", (SELECT COUNT(*) FROM aml_account_alert WHERE account_id = v.id) AS "alertCount", (SELECT COUNT(*) FROM aml_account_alert _e JOIN aml_alerts _tv ON _tv.id = _e.alert_id WHERE _e.account_id = v.id AND _tv.severity = 'HIGH') AS "highSeverity" FROM aml_accounts v WHERE EXISTS (SELECT 1 FROM aml_account_alert we WHERE we.account_id = v.id) LIMIT 20
```

**Result Count:** 1

,accountId,bankId,alertCount,highSeverity
0,100428660,070,1,0


### C8 Cross-bank suspicious flow

Suspicious transfers that cross bank boundaries.


In [21]:
gremlin = "g.E().has('isLaundering','1').project('fromBank','fromAcct','toBank','toAcct','amount','currency').by(outV().values('bankId')).by(outV().values('accountId')).by(inV().values('bankId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C8 Cross-bank suspicious flow', limit=15, tx_mode=False)


### C8 Cross-bank suspicious flow

**Gremlin:**

```groovy
g.E().has('isLaundering','1').project('fromBank','fromAcct','toBank','toAcct','amount','currency').by(outV().values('bankId')).by(outV().values('accountId')).by(inV().values('bankId')).by(inV().values('accountId')).by('amount').by('currency').limit(15)
```

**SQL Translation:**

```sql
SELECT ov.bank_id AS "fromBank", ov.account_id AS "fromAcct", iv.bank_id AS "toBank", iv.account_id AS "toAcct", e.amount AS "amount", e.currency AS "currency" FROM aml_transfers e JOIN aml_accounts ov ON ov.id = e.from_account_id JOIN aml_accounts iv ON iv.id = e.to_account_id WHERE e.is_laundering = ? LIMIT 15
```

**Parameters:** `['1']`

**Result Count:** 5

,fromBank,fromAcct,toBank,toAcct,amount,currency
0,070,100428660,0032375,80E480620,14288.83,US Dollar
1,070,100428660,001124,800825340,389769.39,US Dollar
2,070,100428660,015980,80B39E7B0,792.92,US Dollar
3,070,100428660,011474,805B716C0,29024.33,US Dollar
4,070,100428660,0113798,80DC756E0,13171425.53,US Dollar


### C9 Three-hop money trail

Follow suspicious TRANSFER hops 3 steps deep.


In [22]:
gremlin = "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C9 Three-hop money trail', limit=10, tx_mode=False)


### C9 Three-hop money trail

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(3).path().by('accountId').limit(10)
```

**SQL Translation:**

```sql
SELECT v0.account_id AS accountId0, v1.account_id AS accountId1, v2.account_id AS accountId2, v3.account_id AS accountId3 FROM (SELECT * FROM aml_accounts WHERE EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = id AND we.is_laundering = ?) LIMIT 1) v0 JOIN aml_transfers e1 ON e1.from_account_id = v0.id JOIN aml_accounts v1 ON v1.id = e1.to_account_id JOIN aml_transfers e2 ON e2.from_account_id = v1.id JOIN aml_accounts v2 ON v2.id = e2.to_account_id JOIN aml_transfers e3 ON e3.from_account_id = v2.id JOIN aml_accounts v3 ON v3.id = e3.to_account_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) AND v3.id NOT IN (v0.id, v1.id, v2.id) LIMIT 10
```

**Parameters:** `['1']`

**Result Count:** 10

1. ['100428660', '80BAF4AB0', '80BAF4B00', '80A5A37B0']
2. ['100428660', '80BAF4AB0', '80BAF4B00', '80A5A37B0']
3. ['100428660', '80BAF4AB0', '80BAF4B00', '80A5A37B0']
4. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
5. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
6. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
7. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
8. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
9. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']
10. ['100428660', '8011A2360', '80254DF50', '80E6C15A0']


### C10 Five-hop layering chain

Extended 5-hop traversal for layering detection.


In [23]:
gremlin = "g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)"
result = run_gremlin_query(gremlin, tx_mode=False)
display_query_result(gremlin, result, title='C10 Five-hop layering chain', limit=10, tx_mode=False)


### C10 Five-hop layering chain

**Gremlin:**

```groovy
g.V().hasLabel('Account').where(outE('TRANSFER').has('isLaundering','1')).limit(1).repeat(out('TRANSFER').simplePath()).times(5).path().by('accountId').limit(10)
```

**SQL Translation:**

```sql
SELECT v0.account_id AS accountId0, v1.account_id AS accountId1, v2.account_id AS accountId2, v3.account_id AS accountId3, v4.account_id AS accountId4, v5.account_id AS accountId5 FROM (SELECT * FROM aml_accounts WHERE EXISTS (SELECT 1 FROM aml_transfers we WHERE we.from_account_id = id AND we.is_laundering = ?) LIMIT 1) v0 JOIN aml_transfers e1 ON e1.from_account_id = v0.id JOIN aml_accounts v1 ON v1.id = e1.to_account_id JOIN aml_transfers e2 ON e2.from_account_id = v1.id JOIN aml_accounts v2 ON v2.id = e2.to_account_id JOIN aml_transfers e3 ON e3.from_account_id = v2.id JOIN aml_accounts v3 ON v3.id = e3.to_account_id JOIN aml_transfers e4 ON e4.from_account_id = v3.id JOIN aml_accounts v4 ON v4.id = e4.to_account_id JOIN aml_transfers e5 ON e5.from_account_id = v4.id JOIN aml_accounts v5 ON v5.id = e5.to_account_id WHERE v1.id <> v0.id AND v2.id NOT IN (v0.id, v1.id) AND v3.id NOT IN (v0.id, v1.id, v2.id) AND v4.id NOT IN (v0.id, v1.id, v2.id, v3.id) AND v5.id NOT IN (v0.id, v1.id, v2.id, v3.id, v4.id) LIMIT 10
```

**Parameters:** `['1']`

**Result Count:** 0

### C11 Transactional suspicious count

Suspicious transfer count via transactional endpoint.


In [24]:
gremlin = "g.E().has('isLaundering','1').count()"
result = run_gremlin_query(gremlin, tx_mode=True)
display_query_result(gremlin, result, title='C11 Transactional suspicious count', limit=10, tx_mode=True)


### C11 Transactional suspicious count

**Gremlin:**

```groovy
g.E().has('isLaundering','1').count()
```

**SQL Translation:**

```sql
SELECT COUNT(*) AS count FROM aml_transfers WHERE is_laundering = ?
```

**Parameters:** `['1']`

**Result Count:** 1

1. 5


## Iceberg Note

Use `aml_sql_showcase.ipynb` with `BACKEND='iceberg'` for Iceberg / Trino comparisons.


## Done
